# 03 — Shape & Broadcasting

**Dataset**: `sklearn.datasets.load_digits` — 1797 handwritten digits, 8×8 grayscale images, 10 classes.  
**Goal**: Flatten images, add/remove dimensions, concatenate/stack batches, and use broadcasting  
to subtract a per-pixel mean image — all in both NumPy and PyTorch.

In [ ]:
# ── Shared Setup ────────────────────────────────────────────────────────────
from sklearn.datasets import load_digits
import numpy as np
import torch
import matplotlib.pyplot as plt

digits = load_digits()

np_images = digits.images.astype(np.float32)   # (1797, 8, 8)
np_labels = digits.target.astype(np.int64)     # (1797,)

pt_images = torch.tensor(np_images)
pt_labels = torch.tensor(np_labels)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'images: {np_images.shape} | labels: {np_labels.shape}')

---
## P1 — Flatten 8×8 → 64-d Vectors

Most ML models expect flat feature vectors.  
Flatten the images from `(1797, 8, 8)` to `(1797, 64)` using `reshape` / `view`.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_flat = np_images.reshape(len(np_images), -1)   # (1797, 64)
print('flat:', np_flat.shape)

#### Drill — `flatten_images`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.zeros(2, 3, 3)
# DRILL: flatten keeping batch dim using reshape
flat = t.reshape(2, -1)
assert flat.shape == (2, 9)

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def flatten_images(imgs):
    """
    Flatten (N, 8, 8) → (N, 64).

    Returns:
        t_flat_view    : using .view()
        t_flat_reshape : using .reshape()
        t_flat_flatten : using .flatten(start_dim=1)
    """
    N = imgs.shape[0]

    # .view() requires contiguous — images fresh from tensor() are contiguous
    t_flat_view    = ...   # imgs.view(N, -1)

    # .reshape() makes a copy if needed — safer but may allocate
    t_flat_reshape = ...   # imgs.reshape(N, -1)

    # .flatten(start_dim) is the most readable
    t_flat_flatten = ...   # imgs.flatten(start_dim=1)

    return t_flat_view, t_flat_reshape, t_flat_flatten

t_fv, t_fr, t_ff = flatten_images(pt_images)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
for t in [t_fv, t_fr, t_ff]:
    assert t.shape == (1797, 64)
    assert np.allclose(np_flat, t.numpy())
print('P1 assertions passed ✓')

---
## P2 — Add / Remove Dimensions (Channel Dim)

CNN models expect a channel dimension: `(N, C=1, H, W)`.  
Use `unsqueeze` / `expand_dims` to add it, then `squeeze` to remove it.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_with_channel = np.expand_dims(np_images, axis=1)     # (1797, 1, 8, 8)
np_back         = np.squeeze(np_with_channel, axis=1)   # (1797, 8, 8)

print('with channel:', np_with_channel.shape)
print('squeezed back:', np_back.shape)

#### Drill — `channel_dim_ops`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.zeros(2, 4, 4)
# DRILL: add channel dim at index 1
with_channel = t.unsqueeze(1)
assert with_channel.shape == (2, 1, 4, 4)

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def channel_dim_ops(imgs):
    """
    (N, 8, 8) → (N, 1, 8, 8) → (N, 8, 8)

    Returns:
        t_unsq : (N, 1, 8, 8)
        t_sq   : (N, 8, 8)
    """
    # NumPy equivalent: np.expand_dims(X, axis=1)
    t_unsq = ...   # imgs.unsqueeze(1)

    # NumPy equivalent: np.squeeze(X, axis=1)
    t_sq   = ...   # t_unsq.squeeze(1)

    return t_unsq, t_sq

t_unsq, t_sq = channel_dim_ops(pt_images)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_unsq.shape == (1797, 1, 8, 8)
assert t_sq.shape   == (1797, 8, 8)
assert np.allclose(np_with_channel, t_unsq.numpy())
assert np.allclose(np_back,         t_sq.numpy())
print('P2 assertions passed ✓')

---
## P3 — Transpose & Permute

Transpose a single image (8×8) and permute a batch with a channel dimension.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_T_single = np_images[0].T                                # (8, 8) — 2D transpose

# Batch: (N, 1, H, W) -> (N, H, W, 1) — channels-first to channels-last
np_nchw = np.expand_dims(np_images, axis=1)                 # (1797, 1, 8, 8)
np_nhwc = np_nchw.transpose(0, 2, 3, 1)                    # (1797, 8, 8, 1)

print('single T:', np_T_single.shape)
print('NCHW→NHWC:', np_nhwc.shape)

#### Drill — `transpose_ops`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.zeros(2, 3, 4)
# DRILL: transpose dims 1 and 2
transposed = t.transpose(1, 2)
assert transposed.shape == (2, 4, 3)

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def transpose_ops(imgs):
    """
    Returns:
        t_T       : (8, 8) transposed first image
        t_nhwc    : (N, 8, 8, 1) from (N, 1, 8, 8)
    """
    # 2D transpose — either .T or .transpose(0, 1)
    t_T = ...   # imgs[0].T

    # Batch permute: NCHW → NHWC
    t_nchw = imgs.unsqueeze(1)                 # (N, 1, 8, 8)
    # NumPy equivalent: X.transpose(0, 2, 3, 1)
    t_nhwc = ...   # t_nchw.permute(0, 2, 3, 1)

    return t_T, t_nhwc

t_T, t_nhwc = transpose_ops(pt_images)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_T.shape == (8, 8)
assert np.allclose(np_T_single, t_T.numpy())
assert t_nhwc.shape == (1797, 8, 8, 1)
assert np.allclose(np_nhwc, t_nhwc.numpy())
print('P3 assertions passed ✓')

---
## P4 — Concatenation & Stacking

- `cat` / `concatenate` joins along an **existing** axis.
- `stack` creates a **new** axis.

Simulate building a double-sized batch by concatenating two halves,  
and stacking 10 average-class images into a gallery.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
half = len(np_images) // 2
np_catted = np.concatenate([np_images[:half], np_images[half:]], axis=0)  # (1797, 8, 8)

# Average image per class → stack into (10, 8, 8)
np_means = [np_images[np_labels == c].mean(axis=0) for c in range(10)]
np_stacked = np.stack(np_means, axis=0)   # (10, 8, 8) — new axis

print('catted:', np_catted.shape)
print('stacked class means:', np_stacked.shape)

#### Drill — `cat_and_stack`
Practice the core operation before using it in the problem above.

In [ ]:
a = torch.ones(2)
b = torch.zeros(2)
# DRILL: concat them, then stack them
c = torch.cat([a, b])
s = torch.stack([a, b])
assert c.shape == (4,) and s.shape == (2, 2)

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def cat_and_stack(imgs, labs):
    """
    Returns:
        t_catted  : (N, 8, 8)  — two halves concatenated back
        t_stacked : (10, 8, 8) — per-class mean images stacked
    """
    half = imgs.shape[0] // 2

    # NumPy equivalent: np.concatenate([a, b], axis=0)
    t_catted = ...   # torch.cat([imgs[:half], imgs[half:]], dim=0)

    # Per-class mean images
    means = []
    for c in range(10):
        class_imgs = imgs[labs == c]      # boolean mask
        class_mean = class_imgs.mean(dim=0)   # (8, 8)
        means.append(class_mean)

    # NumPy equivalent: np.stack(list_of_arrays, axis=0)
    t_stacked = ...   # torch.stack(means, dim=0)

    return t_catted, t_stacked

t_catted, t_stacked = cat_and_stack(pt_images, pt_labels)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_catted.shape == np_catted.shape
assert np.allclose(np_catted, t_catted.numpy())
assert t_stacked.shape == (10, 8, 8)
assert np.allclose(np_stacked, t_stacked.numpy(), atol=1e-4)
print('P4 assertions passed ✓')

# Visualize class-mean images
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for c, ax in enumerate(axes.flat):
    ax.imshow(t_stacked[c].numpy(), cmap='gray')
    ax.set_title(f'Mean digit {c}')
    ax.axis('off')
plt.suptitle('Per-class average images')
plt.tight_layout(); plt.show()

---
## P5 — Broadcasting: Subtract the Mean Image

Compute the **global mean image** `(8, 8)` across all 1797 samples, then subtract it  
from every image using broadcasting: `(1797, 8, 8) - (8, 8)` → `(1797, 8, 8)`.

This is mean-centering — one of the most common preprocessing steps.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_mean_img  = np_images.mean(axis=0)                        # (8, 8)
np_centered  = np_images - np_mean_img                       # broadcast! (1797,8,8) - (8,8)

print('mean image shape:', np_mean_img.shape)
print('centered mean (should ≈ 0):', np_centered.mean(axis=0).max())

#### Drill — `mean_center`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.ones(2, 3)
# DRILL: subtract scalar mean via broadcasting
centered = t - t.mean()
assert centered.sum() == 0

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def mean_center(imgs):
    """
    Subtract the global mean image from every image.

    Returns:
        t_mean_img : (8, 8)       — the mean image
        t_centered : (1797, 8, 8) — mean-centered images
    """
    # NumPy equivalent: X.mean(axis=0)
    t_mean_img = ...   # imgs.mean(dim=0)   → shape (8, 8)

    # Broadcasting: (1797, 8, 8) - (8, 8) → broadcasts (8,8) across dim 0
    t_centered = ...   # imgs - t_mean_img

    return t_mean_img, t_centered

t_mean_img, t_centered = mean_center(pt_images)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_mean_img.shape == (8, 8)
assert t_centered.shape == (1797, 8, 8)
assert np.allclose(np_mean_img, t_mean_img.numpy(),   atol=1e-5)
assert np.allclose(np_centered, t_centered.numpy(),   atol=1e-4)
# The mean of centered images along axis 0 should be ≈ 0
assert np.allclose(t_centered.mean(dim=0).numpy(), 0, atol=1e-4)
print('P5 assertions passed ✓')

---
## P6 — Broadcasting: Per-Image Feature Normalization

Normalize each flattened image independently to zero mean and unit std.  
This requires `keepdim=True` to maintain broadcast-compatible shapes.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_flat = np_images.reshape(len(np_images), -1)         # (1797, 64)
np_per_mean = np_flat.mean(axis=1, keepdims=True)       # (1797, 1)
np_per_std  = np_flat.std(axis=1, keepdims=True)        # (1797, 1)
np_normed   = (np_flat - np_per_mean) / (np_per_std + 1e-8)  # (1797, 64)

print('per-image normalized mean (sample 0):', np_normed[0].mean().round(6))
print('per-image normalized std  (sample 0):', np_normed[0].std().round(4))

#### Drill — `per_image_normalize`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.arange(4, dtype=torch.float32).reshape(2, 2)
# DRILL: divide by max of each row using keepdim
max_vals = t.max(dim=1, keepdim=True).values
norm = t / max_vals
assert norm.shape == (2, 2) and norm[1, 1] == 1.0

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def per_image_normalize(imgs):
    """
    Normalize each image (row) of the flattened (N, 64) tensor independently.

    Returns:
        t_normed : (N, 64)
    """
    t_flat = imgs.flatten(start_dim=1)            # (N, 64)

    # NumPy equivalent: X.mean(axis=1, keepdims=True)
    t_mean = ...   # t_flat.mean(dim=1, keepdim=True)   → (N, 1)

    # NOTE: use correction=0 (PyTorch default is Bessel's) to match NumPy default
    t_std  = ...   # t_flat.std(dim=1, keepdim=True, correction=0)  → (N, 1)

    # Broadcasting: (N, 64) - (N, 1) → (N, 64)
    t_normed = ...  # (t_flat - t_mean) / (t_std + 1e-8)

    return t_normed

t_normed = per_image_normalize(pt_images)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_normed.shape == (1797, 64)
assert np.allclose(np_normed, t_normed.numpy(), atol=1e-4)
print('P6 assertions passed ✓')